# Chapter 16: The Kalman Filter

<a href="../lite/lab/index.html?path=ch16_kalman_filter.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite — run and edit this notebook</a>

*Runs entirely in your browser — no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 16.1 The Linear-Gaussian Assumption

The Kalman filter operates on a system defined by two models.

**Motion model** (how the state evolves):
$$\mathbf{x}_t = F\mathbf{x}_{t-1} + B\mathbf{u}_t + \mathbf{w}_t, \quad \mathbf{w}_t \sim \mathcal{N}(0, Q)$$

**Observation model** (how sensors relate to state):
$$\mathbf{z}_t = H\mathbf{x}_t + \mathbf{v}_t, \quad \mathbf{v}_t \sim \mathcal{N}(0, R)$$

$Q$ is the process noise covariance (how much we trust our motion model) and $R$ is the measurement noise covariance (how much we trust our sensors).

## 16.2 Prediction Step

$$\bar{\boldsymbol{\mu}}_t = F\boldsymbol{\mu}_{t-1} + B\mathbf{u}_t \qquad \bar{\boldsymbol{\Sigma}}_t = F\boldsymbol{\Sigma}_{t-1}F^\top + Q$$

Prediction **grows** the uncertainty — motion adds process noise.

## 16.3 Update Step

$$K_t = \bar{\boldsymbol{\Sigma}}_t H^\top (H\bar{\boldsymbol{\Sigma}}_t H^\top + R)^{-1}$$
$$\boldsymbol{\mu}_t = \bar{\boldsymbol{\mu}}_t + K_t(\mathbf{z}_t - H\bar{\boldsymbol{\mu}}_t) \qquad \boldsymbol{\Sigma}_t = (I - K_t H)\bar{\boldsymbol{\Sigma}}_t$$

Update **shrinks** the uncertainty — a measurement gives information.

## 16.4 The Kalman Gain: Intuition

$K$ balances trust between prediction and measurement. When $R$ is large (noisy sensor), $K$ is small and the filter trusts the prediction. When $R$ is small (accurate sensor), $K$ is large and the filter trusts the measurement. The term $(\mathbf{z}_t - H\bar{\boldsymbol{\mu}}_t)$ is the **innovation** — how surprised we are by the measurement.

## 16.5 Covariance Convergence

**Try it:** Change the noise parameters and observe how the filter behaves.

```{admonition} What you will build
:class: tip

- Implement a 1D Kalman filter and watch its uncertainty converge
- Build a 2D Kalman filter that tracks a robot using noisy position measurements
- Understand the Kalman gain as the optimal balance between prediction and measurement
- Visualize the covariance ellipse shrinking as measurements arrive

**Real world application:** The Kalman filter runs in GPS receivers, autopilots, and smartphone sensors. After this chapter, you will have implemented the most widely used estimation algorithm in engineering.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **FilterPy** | Python Kalman filter library (`KalmanFilter` class) |
| **robot_localization (ROS 2)** | Production EKF/UKF for fusing IMU, odometry, GPS in ROS |
| **Eigen + custom** | Most production Kalman filters are hand-written in C++ with Eigen for performance |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
process_noise_std     = 0.5   # how much the motion model drifts  (try 0.1, 1.0, 2.0)
measurement_noise_std = 1.5   # how noisy the sensor is           (try 0.2, 1.5, 4.0)
n_steps               = 40    # number of time steps               (try 20, 60, 100)
# ────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
Q = process_noise_std**2
R = measurement_noise_std**2

true_positions = np.cumsum(np.ones(n_steps))
true_positions += np.random.normal(0, process_noise_std, n_steps).cumsum()
measurements = true_positions + np.random.normal(0, measurement_noise_std, n_steps)

mu, sigma2 = 0.0, 1.0
estimates, variances = [], []
for t in range(n_steps):
    mu     = mu + 1.0              # predict
    sigma2 = sigma2 + Q
    K      = sigma2 / (sigma2 + R) # update
    mu     = mu + K * (measurements[t] - mu)
    sigma2 = (1 - K) * sigma2
    estimates.append(mu); variances.append(sigma2)

estimates = np.array(estimates)
stds = np.sqrt(np.array(variances))
steps = np.arange(1, n_steps+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(steps, true_positions, 'k-', lw=2, label='True position', zorder=3)
ax1.scatter(steps, measurements, s=20, color='tomato', alpha=0.5, label='Measurements', zorder=2)
ax1.plot(steps, estimates, 'steelblue', lw=2, label='KF estimate', zorder=4)
ax1.fill_between(steps, estimates-2*stds, estimates+2*stds, alpha=0.2, color='steelblue', label='±2σ')
ax1.set_xlabel('Time step'); ax1.set_ylabel('Position (m)')
ax1.set_title('Kalman Filter: Position Estimate'); ax1.legend(fontsize=8)

ax2.plot(steps, np.array(variances), 'steelblue', lw=2)
ax2.axhline(Q, color='orange', linestyle='--', alpha=0.7, label=f'Process noise Q={Q:.2f}')
ax2.axhline(R, color='tomato',  linestyle='--', alpha=0.7, label=f'Sensor noise R={R:.2f}')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Variance σ²')
ax2.set_title('Covariance Convergence'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f'Steady-state variance:  {variances[-1]:.4f}')
print(f'Kalman gain (final):    {variances[-1]/(variances[-1]+R):.4f}')

**What to notice:**
1. **Large process noise, small sensor noise** — the filter trusts measurements; estimate tracks them closely.
2. **Small process noise, large sensor noise** — the filter trusts its own motion model; estimate is smoother.
3. **Covariance convergence** — the right plot shows variance settling to a steady state regardless of the initial value.

## 16.6 Implementation: 2D Kalman Filter

The state is $[x, y, \dot{x}, \dot{y}]$ — position and velocity. The sensor only observes position.

In [ ]:
class KalmanFilter2D:
    """
    2D Kalman filter: state = [x, y, vx, vy], observation = [x, y]
    """
    def __init__(self, dt, process_noise, measurement_noise):
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]])
        self.H = np.array([[1,0,0,0],[0,1,0,0]])
        self.Q = (process_noise**2) * np.eye(4)
        self.R = (measurement_noise**2) * np.eye(2)
        self.mu    = np.zeros(4)
        self.Sigma = np.eye(4) * 5.0

    def predict(self):
        self.mu    = self.F @ self.mu
        self.Sigma = self.F @ self.Sigma @ self.F.T + self.Q

    def update(self, z):
        S = self.H @ self.Sigma @ self.H.T + self.R
        K = self.Sigma @ self.H.T @ np.linalg.inv(S)
        self.mu    = self.mu + K @ (z - self.H @ self.mu)
        self.Sigma = (np.eye(4) - K @ self.H) @ self.Sigma

    @property
    def position(self):     return self.mu[:2]
    @property
    def position_cov(self): return self.Sigma[:2, :2]

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
process_noise     = 0.3   # motion model uncertainty  (try 0.05, 0.5, 1.5)
measurement_noise = 0.8   # sensor noise              (try 0.1,  0.8, 2.5)
n_steps           = 60    # trajectory length         (try 30, 60, 120)
# ────────────────────────────────────────────────────────────────────────────

np.random.seed(7)
dt = 0.1
t = np.linspace(0, n_steps*dt, n_steps)
true_x = 5*np.sin(0.5*t);  true_y = 0.3*t**1.3
meas_x = true_x + np.random.normal(0, measurement_noise, n_steps)
meas_y = true_y + np.random.normal(0, measurement_noise, n_steps)

kf = KalmanFilter2D(dt, process_noise, measurement_noise)
est_pos, est_covs = [], []
for i in range(n_steps):
    kf.predict()
    kf.update(np.array([meas_x[i], meas_y[i]]))
    est_pos.append(kf.position.copy())
    est_covs.append(kf.position_cov.copy())
est_pos = np.array(est_pos)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(true_x, true_y, 'k-', lw=2.5, label='True trajectory', zorder=5)
ax.scatter(meas_x, meas_y, s=15, color='tomato', alpha=0.4, label='Measurements', zorder=3)
ax.plot(est_pos[:,0], est_pos[:,1], 'steelblue', lw=2, label='KF estimate', zorder=4)

for i in range(0, n_steps, max(1, n_steps//12)):
    ev, evec = np.linalg.eigh(est_covs[i])
    ang = np.degrees(np.arctan2(evec[1,1], evec[0,1]))
    ell = patches.Ellipse(est_pos[i], 2*2*np.sqrt(max(ev[1],1e-9)), 2*2*np.sqrt(max(ev[0],1e-9)),
                          angle=ang, facecolor='steelblue', edgecolor='steelblue', alpha=0.15)
    ax.add_patch(ell)

ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('2D Kalman Filter  (ellipses = 2σ uncertainty at each step)')
ax.legend(); plt.tight_layout(); plt.show()

**What to observe:** The ellipses shrink as the filter gains confidence. High measurement noise makes them larger. High process noise makes the estimate follow measurements more aggressively.

## Exercises

**Exercise 16.1:** Modify `KalmanFilter2D.__init__` so position and velocity have *different* process noise. Use `np.diag([q_pos, q_pos, q_vel, q_vel])` for Q. Does tracking improve on the curve?

In [ ]:
# Your code here

**Exercise 16.2:** What happens if the filter starts with a very wrong position (set `self.mu = np.array([100, 100, 0, 0])`)? How quickly does it recover? What does this tell you about the initial covariance?

In [ ]:
# Your code here

**Exercise 16.3 (challenge):** Simulate **measurement dropout** — some fraction of steps have no measurement. In those steps, only call `predict()`, not `update()`. How does the uncertainty ellipse grow during a dropout period?

In [ ]:
# Your code here